# Graph Animation demos

In Colab, run the Manim install cell first and restart the runtime when it finishes. Then run the load-project cell, the Manim extension cell, and whichever demo scenes you want to render. This notebook is opened from GitHub, so the project source is cloned into the runtime automatically.


In [ ]:
# Colab setup. If you already have Manim installed locally with its native dependencies, skip this cell.
import subprocess
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "libcairo2-dev", "libpango1.0-dev", "ffmpeg"], check=True)
%pip install -q manim==0.20.1 ipython==8.21.0
print("If using Colab, restart the runtime now, then continue with the load-project cell.")


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "graph_animation").is_dir():
    ROOT = ROOT / "graph-animation-repository"
    if not (ROOT / "graph_animation").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/shrnasa011/graph_animation.git", str(ROOT)
        ], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [1]:
%load_ext manim

The manim module is not an IPython extension.


In [2]:
from graph_animation.graph import Graph

In [ ]:
%%manim -v WARNING -pql -t GraphDemo

from manim import *
from graph_animation.graph import Graph
import numpy as np


class GraphDemo(Scene):
    def construct(self):
        self.camera.background_color = WHITE
        
        positions = {
            "a": np.array([-2, 1, 0]),
            "b": np.array([0, 2, 0]),
            "c": np.array([-2, -1, 0]),
            "d": np.array([1.5, 0, 0]),
            "e": np.array([3, 1.5, 0]),
        }

        graph = Graph(
            vertices=["a", "b", "c", "d", "e"],
            edges=[
                ("a", "b"),
                ("a", "c"),
                ("b", "d"),
                ("c", "d"),
                ("d", "e"),
            ],
            positions=positions,
        )

        graph.view.set_width(config.frame_width * 0.5)
        self.wait(0.2)
        self.add(graph.view)
        self.wait(0.5)

        self.play(graph.show_degrees())

        self.wait(0.5)

        self.play(graph.show_halo("d"))

        self.wait(0.5)

        self.play(graph.select("d"))

        self.wait(0.5)

        self.play(graph.gray_vertices(["b", "c"]))

        self.wait(0.5)

        self.play(graph.update_degrees())
        
        self.wait(0.5)

        self.play(graph.hide_degrees())

        self.wait(0.5)

        self.play(graph.hide_halo("d"))

        self.wait()

In [4]:
%%manim -v WARNING -qh -t GraphContainer

"""
Demo animation built on top of the ``graph_animation`` package.

Run it with::

    manim -pql demo.py GraphDemo      # quick preview (low quality)
    manim -qh demo.py GraphDemo       # high quality render

The scene walks a tiny greedy "cover" style animation:
    1.  fade the graph in
    2.  show every vertex degree
    3.  pick the highest-degree vertex, select it and light up its edges
    4.  gray out that vertex and its neighbours, then refresh the degrees
    5.  reset everything
"""

from __future__ import annotations

import os
import sys

import numpy as np
from manim import *

# Make the ``graph_animation`` package importable regardless of where
# manim is invoked from (the package uses relative imports internally).
# sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from graph_animation import Graph

from pathlib import Path

def save_frame(self, name):
    out_dir = Path("frames_container_graphs")
    out_dir.mkdir(exist_ok=True)

    self.renderer.update_frame(self)
    self.camera.get_image().save(out_dir / f"{name}.png")

def update_visible_degrees(graph):
    """Refresh active degrees and keep inactive labels hidden."""
    animations = []
    for name, vertex in graph.vertices.items():
        if vertex.is_active():
            animations.append(
                vertex.degree_label.animate.become(
                    vertex.degree_target(graph.degree(name))
                )
            )
        else:
            animations.append(vertex.degree_label.animate.set_opacity(0))
    return AnimationGroup(*animations)

class GraphContainer(Scene):
    def construct(self):
        # White canvas.
        self.camera.background_color = WHITE

        # ------------------------------------------------------------------
        # 1. Define the graph: vertices, edges and 2-D layout positions.
        # ------------------------------------------------------------------
        positions = {
            "a": np.array([-3.0,  1.5, 0.0]),
            "b": np.array([-1.0,  2.2, 0.0]),
            "c": np.array([ 1.0,  1.5, 0.0]),
            "d": np.array([ 3.0,  0.5, 0.0]),
            "e": np.array([-2.0, -0.5, 0.0]),
            "f": np.array([ 0.0,  0.2, 0.0]),
            "g": np.array([ 2.0, -1.2, 0.0]),
            "h": np.array([ 4.0, -0.5, 0.0]),
        }

        edges = [
            ("a", "b"),
            ("b", "c"),
            ("c", "d"),
            ("a", "e"),
            ("b", "f"),
            ("c", "f"),
            ("e", "f"),
            ("f", "g"),
            ("d", "g"),
            ("d", "h"),
            ("h", "g"),
        ]

        graph = Graph(
            vertices=list(positions.keys()),
            edges=edges,
            positions=positions,
        )

        # Scale the whole view down; the degree labels created later
        # will match this scale automatically.
        graph.scale(1.5)

        # ------------------------------------------------------------------
        # 2. Fade the graph in.
        # ------------------------------------------------------------------
        title = Text(
            "",
            font_size=32,
            color=BLACK,
        ).to_edge(UP)
        graph.add_to_scene(self)   # add so edge updaters stay live
        # self.play(FadeIn(title))
        self.play(FadeIn(graph.mobject))
        self.wait(0.5)

        # ------------------------------------------------------------------
        # 3b. Scale the whole graph view up and back down.
        #     The degree labels (children of the view VGroup) scale
        #     right along with the vertices and edges.
        # ------------------------------------------------------------------
        # self.play(graph.mobject.animate.scale(1.6), run_time=1.5)
        # self.wait(0.5)

        
        # ------------------------------------------------------------------
        # 3. Reveal every vertex degree.
        # ------------------------------------------------------------------
        self.play(graph.show_degrees())
        self.wait(0.5)

        f = 1
        save_frame(self, f)
        f += 1
        
        self.renderer.file_writer.save_final_image = True
        self.renderer.file_writer.save_png = True
        # self.play(graph.mobject.animate.scale(1 / 1.6), run_time=1.5)
        # self.wait(0.5)

        # Earlier vertices have priority when two degrees are equal.
        input_order = ["a", "b", "c", "d", "e", "f", "g", "h"]
        order_rank = {vertex: rank for rank, vertex in enumerate(input_order)}
        I = {"b", "d"}
        removed = set()
        # ------------------------------------------------------------------
        # 4. Greedily pick the highest-degree vertex.
        # ------------------------------------------------------------------
        for i in range(1,3):
            candidates = [
                vertex for vertex in input_order
                if vertex in I and vertex not in removed
            ]
            if not candidates:
                break
            chosen = max(
                candidates,
                key=lambda vertex: (graph.degree(vertex), -order_rank[vertex]),
            )
            removed.add(chosen)
            self.play(graph.pulse(chosen))
            self.play(
                graph.show_halo(chosen),
                graph.select(chosen),
            )
            save_frame(self, f)
            f += 1
            
            if graph.degree(chosen) == 0:
                continue
    
            # Remove active vertices whose current degree is larger than
            # the chosen vertex's degree before handling its neighbours.
            chosen_degree = graph.degree(chosen)
            high_degree = [
                vertex
                for vertex in graph
                if vertex != chosen
                and graph.is_active(vertex)
                and graph.degree(vertex) > chosen_degree
            ]
            if high_degree:
                self.play(
                    graph.gray_vertices(high_degree),
                    *[
                        graph[vertex].degree_label.animate.set_opacity(0)
                        for vertex in high_degree
                    ],
                    run_time=0.45,
                )
                self.play(update_visible_degrees(graph), run_time=0.35)
                self.wait(1.0)

            save_frame(self, f)
            f += 1
            print("High degree frame " + str(f))

            # Light up active incident edges, if any remain.
            incident_edge_animations = [
                graph.highlight_edge(chosen, neighbour)
                for neighbour in graph.neighbors(chosen)
                if graph.is_active(neighbour)
            ]
            if incident_edge_animations:
                self.play(
                    AnimationGroup(*incident_edge_animations, lag_ratio=0.25)
                )
            self.wait(1.0)
            
            # Gray only neighbours that remain active after the
            # higher-degree vertices have been removed.
            neighbours = [
                vertex for vertex in graph.neighbors(chosen)
                if graph.is_active(vertex)
            ]
    
            removal_animations = [
                graph.hide_halo(chosen),
                graph.gray(chosen),
                graph[chosen].degree_label.animate.set_opacity(0),
            ]
            if neighbours:
                removal_animations.append(graph.gray_vertices(neighbours))
                removal_animations.extend(
                    graph[vertex].degree_label.animate.set_opacity(0)
                    for vertex in neighbours
                )
            self.play(*removal_animations)
            self.wait(0.5)
            self.play(update_visible_degrees(graph))
            self.wait(0.5)
            
            save_frame(self, f)
            f += 1

        # ------------------------------------------------------------------
        # 6. Reset the graph back to its original state.
        # ------------------------------------------------------------------
        # self.play(graph.hide_degrees())
        # self.play(graph.reset())
        # self.wait(1.0)


Manim Community v0.20.1

[08/15/26 12:20:51] WARNING  Output format changed to '.webm' to support transparency                 ]8;id=1119184;file:///opt/anaconda3/envs/jupytern/lib/python3.11/site-packages/manim/_config/utils.py\utils.py]8;;\:]8;id=1119185;file:///opt/anaconda3/envs/jupytern/lib/python3.11/site-packages/manim/_config/utils.py#1449\1449]8;;\

                    WARNING  Output format set as webm, this can be slower than other formats         ]8;id=1119190;file:///opt/anaconda3/envs/jupytern/lib/python3.11/site-packages/manim/_config/utils.py\utils.py]8;;\:]8;id=1119191;file:///opt/anaconda3/envs/jupytern/lib/python3.11/site-packages/manim/_config/utils.py#1063\1063]8;;\

High degree frame 4


High degree frame 7


In [5]:
%%manim -v WARNING -qh -t --disable_caching TriHyperGraphDemo

"""
3-uniform hypergraph demo.

Every hyperedge joins exactly three vertices and is drawn as a rounded
triangle (TriEdge). Render with::

    .venv/bin/manim -ql --disable_caching demo_tri_hypergraph.py TriHyperGraphDemo
    .venv/bin/manim -qh demo_tri_hypergraph.py TriHyperGraphDemo
"""

from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
from manim import *

# sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from graph_animation import TriHyperGraph
from graph_animation.primitives import Edge

def save_frame(self, name):
    out_dir = Path("frames_container_hypergraphs")
    out_dir.mkdir(exist_ok=True)

    self.renderer.update_frame(self)
    self.camera.get_image().save(out_dir / f"{name}.png")

def update_visible_hypergraph_degrees(hypergraph):
    """Refresh active degrees while keeping inactive labels hidden."""
    animations = []
    for name, vertex in hypergraph.vertices.items():
        if vertex.is_active():
            animations.append(
                vertex.degree_label.animate.become(
                    vertex.degree_target(hypergraph.degree(name))
                )
            )
        else:
            animations.append(vertex.degree_label.animate.set_opacity(0))
    return AnimationGroup(*animations)

class TriHyperGraphDemo(Scene):
    def construct(self):
        self.camera.background_color = WHITE

        positions = {
            "1": np.array([-3.2,  1.6, 0]),
            "2": np.array([-1.2,  2.2, 0]),
            "3": np.array([-2.6, -0.6, 0]),
            "4": np.array([-0.4,  0.2, 0]),
            "5": np.array([ 1.8,  1.7, 0]),
            "6": np.array([ 2.6, -0.6, 0]),
            "7": np.array([ 0.6, -1.6, 0]),
            "8": np.array([ 4.0, 0, 0]),
        }

        # Every hyperedge has exactly three vertices (3-uniform).
        hyperedges = [
            ("1", "2", "3"),
            ("2", "3", "4"),
            ("4", "5", "6"),
            ("4", "6", "7"),
            ("5", "6", "8"),
        ]

        hg = TriHyperGraph(
            vertices=list(positions.keys()),
            hyperedges=hyperedges,
            positions=positions,
        )

        hg.view.width = config.frame_width * 0.6
        hg.view.move_to(0.5 * DOWN)

        # title = Text("3-uniform hypergraph", font_size=34, color=BLACK).to_edge(UP)
        caption = Text(
            "",
            font_size=24, color=GREY_D,
        ).to_edge(DOWN)

        # 1. Bring in the vertices, then fade the triangles in around them.
        # self.play(FadeIn(title))
        self.play(FadeIn(hg.vertex_group, lag_ratio=0.08), run_time=1.0)
        self.wait(0.3)
        self.play(hg.draw_hyperedges(lag_ratio=0.35), run_time=1.6)
        self.wait(0.5)

        # 2. Show incidence degrees.
        self.play(FadeIn(caption), hg.show_degrees(), run_time=0.8)
        self.wait(0.8)

        f = 1
        save_frame(self, f)
        f += 1
        
        # # 3. Gently light up each triangle in turn.
        # for i in range(len(hyperedges)):
        #     self.play(hg.highlight_hyperedge(i), run_time=0.6)
        #     self.wait(0.4)
        #     self.play(hg.unhighlight_hyperedge(i), run_time=0.6)

        # self.wait(0.3)

        input_order = ["1", "2", "3", "4", "5", "6", "7", "8"]
        order_rank = {vertex: rank for rank, vertex in enumerate(input_order)}
        I = {"2", "5", "7", "8"}
        for _ in range(3):
            # First select the maximum-degree vertex in the active
            # 3-uniform hypergraph (restricted to the input set I).
            candidates = [
                v for v in input_order if v in I and hg[v].is_active()
            ]
            if not candidates:
                break
            hypergraph_vertex = max(
                candidates,
                key=lambda v: (hg.degree(v), -order_rank[v]),
            )
            self.play(hg.pulse(hypergraph_vertex))
            self.play(hg.show_halo(hypergraph_vertex), hg.select(hypergraph_vertex))
            self.wait(0.5)
            save_frame(self, f)
            f += 1

            # Remove active vertices of larger hypergraph degree before
            # constructing the link graph at this nested level.
            hypergraph_degree = hg.degree(hypergraph_vertex)
            high_hypergraph_vertices = [
                vertex for vertex in hg
                if vertex != hypergraph_vertex
                and hg[vertex].is_active()
                and hg.degree(vertex) > hypergraph_degree
            ]
            if high_hypergraph_vertices:
                self.play(
                    hg.remove_vertices(high_hypergraph_vertices),
                    *[
                        hg[vertex].degree_label.animate.set_opacity(0)
                        for vertex in high_hypergraph_vertices
                    ],
                    run_time=0.6,
                )
                self.play(update_visible_hypergraph_degrees(hg), run_time=0.45)

            self.wait(1.0)
            save_frame(self, f)
            f += 1

            # Form the link graph: every triple through hypergraph_vertex
            # produces the 2-edge left after removing that vertex.
            link_degrees = hg.link_graph_degrees(hypergraph_vertex)
            self.play(hg.fade(), run_time=0.35)
            link_graph = VGroup()
            projected_edges = []
            link_annotations = VGroup()
            for v, degree in link_degrees.items():
                # The original labels are dimmed with the hypergraph, so
                # copy them into the foreground with purple link degrees.
                vertex_label = hg[v].label.copy().set_color(BLACK).set_opacity(1)
                degree_label = MathTex(f"[{degree}]", color="#1F2A5A")
                degree_label.match_height(hg[v].degree_label)
                degree_label.next_to(hg[v].circle, DOWN, buff=0.12)
                link_annotations.add(vertex_label, degree_label)
            link_caption = Text(
                "",
                font_size=20, color=PURPLE_D,
            ).to_edge(DOWN)
            for hyperedge_index, (u, v) in hg.link_graph_edges(hypergraph_vertex):
                projected_edge = Edge(hg[u], hg[v], color=YELLOW, stroke_width=6)
                link_graph.add(projected_edge)
                projected_edges.append((projected_edge, (u, v)))
                self.play(
                    hg.highlight_hyperedge(hyperedge_index),
                    Create(projected_edge),
                    run_time=0.55,
                )

            self.play(FadeIn(link_annotations), FadeIn(link_caption), run_time=0.35)

            self.wait(1.0)
            save_frame(self, f)
            f += 1
            
            # Next select the maximum-degree link-graph vertex that is
            # still available in the input set I.
            link_candidates = [
                v for v in input_order
                if v in I and v in link_degrees and hg[v].is_active()
            ]
            link_vertex = (
                max(
                    link_candidates,
                    key=lambda v: (link_degrees[v], -order_rank[v]),
                )
                if link_candidates else None
            )
            # print(link_vertex)
            # print(len(link_candidates))
            if link_vertex is not None:
                self.play(hg.pulse(link_vertex))
                self.play(hg.show_halo(link_vertex), hg.select(link_vertex))
                self.play(
                    *[edge.highlight() for edge, ends in projected_edges if link_vertex in ends],
                    run_time=0.45,
                )
                self.wait(1.0)
                save_frame(self, f)
                f += 1
                high_link_vertices = [
                    vertex for vertex, degree in link_degrees.items()
                    if vertex != link_vertex
                    and hg[vertex].is_active()
                    and degree > link_degrees[link_vertex]
                ]
                link_neighbours = {
                    v if u == link_vertex else u
                    for _, (u, v) in hg.link_graph_edges(hypergraph_vertex)
                    if link_vertex in (u, v)
                }
            else:
                high_link_vertices = []
                link_neighbours = set()

            self.wait(0.5)
            self.play(
                FadeOut(link_graph),
                FadeOut(link_annotations),
                FadeOut(link_caption),
                hg.unfade(),
                run_time=0.55,
            )

            # At the link-graph level, remove higher-degree vertices
            # before removing the two selected vertices.
            if high_link_vertices:
                self.play(
                    hg.remove_vertices(high_link_vertices),
                    *[
                        hg[vertex].degree_label.animate.set_opacity(0)
                        for vertex in high_link_vertices
                    ],
                    run_time=0.6,
                )
                self.play(update_visible_hypergraph_degrees(hg), run_time=0.45)
            self.wait(1.0)
            save_frame(self, f)
            f += 1

            # Remove both selected vertices, then remove the still-active
            # neighbours of the selected link-graph vertex as well.
            removed_vertices = [hypergraph_vertex]
            if link_vertex is not None:
                removed_vertices.append(link_vertex)
            removed_vertices.extend(
                vertex for vertex in link_neighbours if hg[vertex].is_active()
            )
            self.play(
                *[hg.hide_halo(v) for v in removed_vertices],
                *[hg.deselect(v) for v in removed_vertices],
                run_time=0.35,
            )
            self.play(hg.remove_vertices(removed_vertices), run_time=0.6)
            self.play(update_visible_hypergraph_degrees(hg), run_time=0.5)
            self.wait(0.5)
            save_frame(self, f)
            f += 1
        
        # 4. Vertex "4" belongs to three triples — spotlight it together
        #    with the triangles that contain it.
        # self.play(
        #     hg.select("4"),
        #     hg.show_halo("4"),
        #     *[hg.highlight_hyperedge(i) for i in hg.incidence["4"]],
        #     run_time=0.9,
        # )
        # self.wait(1.0)

        # # 5. Release the spotlight.
        # self.play(
        #     hg.deselect("4"),
        #     hg.hide_halo("4"),
        #     *[hg.unhighlight_hyperedge(i) for i in hg.incidence["4"]],
        #     run_time=0.9,
        # )
        # self.wait(1.0)


Manim Community v0.20.1

[08/17/26 16:45:56] WARNING  Output format changed to '.webm' to support transparency                 ]8;id=1119196;file:///opt/anaconda3/envs/jupytern/lib/python3.11/site-packages/manim/_config/utils.py\utils.py]8;;\:]8;id=1119197;file:///opt/anaconda3/envs/jupytern/lib/python3.11/site-packages/manim/_config/utils.py#1449\1449]8;;\

                    WARNING  Output format set as webm, this can be slower than other formats         ]8;id=1119202;file:///opt/anaconda3/envs/jupytern/lib/python3.11/site-packages/manim/_config/utils.py\utils.py]8;;\:]8;id=1119203;file:///opt/anaconda3/envs/jupytern/lib/python3.11/site-packages/manim/_config/utils.py#1063\1063]8;;\